# ATDL Phase 8 — Comprehensive Model and Imbalance Evaluation

This notebook runs the full-data pretrained-model benchmark, validation-only candidate selection, equal-exposure tail-class interventions, one final held-out test, and report/figure generation. Phase 8 training and model selection use only train and validation data. The same split was evaluated for the historical YOLOv8n baseline in Phase 6, so it is not a project-wide blind holdout.

## Study question
Can modern COCO-pretrained YOLO models and tail-aware training improve AP50/recall for `crazing`, `pitted_surface`, `rolled-in_scale`, and `scratches` while maintaining overall defect detection performance?

## Run order
1. Finish Phases 1–7 and use the project `.venv` kernel.
2. Run the four-architecture, three-seed validation benchmark.
3. Compare 320 px and 640 px inputs for the selected architecture across paired seeds.
4. Run full-data repeat sampling, targeted augmentation, and combined treatment.
5. Select the candidate by validation only and inspect the validation figures.
6. Run the final test cell once. Do not rerun it to choose another model.
7. Build the report files and slides from the saved results.

Runtime is GPU dependent and is expected to take multiple hours. Experiments resume from verified checkpoints and metric files when rerun without `--force`.

## 1. Project root and runtime
Use the repository `.venv` as the Jupyter kernel in VS Code.

In [1]:
from pathlib import Path
import json, subprocess, sys

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'configs/neu_det_yolo.yaml').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open the atdl_final_project repository in VS Code.')
print('Project:', PROJECT_ROOT)
print('Python:', sys.executable)
assert (PROJECT_ROOT / 'data/processed/images/train').is_dir()
assert (PROJECT_ROOT / 'data/processed/images/val').is_dir()
assert (PROJECT_ROOT / 'data/processed/images/test').is_dir()

Project: /home/vsu/Desktop/atdl_final_project
Python: /home/vsu/Desktop/atdl_final_project/.venv/bin/python


## 2. Architecture transfer benchmark
YOLOv8n, YOLO11n, YOLO26n, and YOLO11s use the same fixed Phase 2 train/validation split, image size, batch size, epochs, optimizer configuration, and seeds 42–44. The existing YOLOv8n seed-42 baseline is reused. Only validation metrics are read.

In [2]:
architecture_command = [sys.executable, '-m', 'src.atdl.run_comprehensive_study', '--stage', 'architectures']
subprocess.run(architecture_command, cwd=PROJECT_ROOT, check=True)
architecture_summary = PROJECT_ROOT / 'results/comprehensive/model_transfer_summary.csv'
print(architecture_summary.read_text())


[2026-11-19T07:34:03.609036+00:00] yolov8n seed 42
completed yolov8n seed 42: tail AP50=0.6933, mAP50=0.7578

[2026-11-19T07:34:03.749788+00:00] yolov8n seed 43
completed yolov8n seed 43: tail AP50=0.6482, mAP50=0.7244



[2026-11-19T07:34:03.865848+00:00] yolov8n seed 44
completed yolov8n seed 44: tail AP50=0.6500, mAP50=0.7254

[2026-11-19T07:34:03.980741+00:00] yolo11n seed 42
completed yolo11n seed 42: tail AP50=0.6684, mAP50=0.7406



[2026-11-19T07:34:04.090073+00:00] yolo11n seed 43
completed yolo11n seed 43: tail AP50=0.6682, mAP50=0.7407

[2026-11-19T07:34:04.198717+00:00] yolo11n seed 44
completed yolo11n seed 44: tail AP50=0.6592, mAP50=0.7323



[2026-11-19T07:34:04.306816+00:00] yolo26n seed 42
completed yolo26n seed 42: tail AP50=0.6698, mAP50=0.7409

[2026-11-19T07:34:04.414569+00:00] yolo26n seed 43
completed yolo26n seed 43: tail AP50=0.6928, mAP50=0.7568



[2026-11-19T07:34:04.522377+00:00] yolo26n seed 44
completed yolo26n seed 44: tail AP50=0.6745, mAP50=0.7489

[2026-11-19T07:34:04.630040+00:00] yolo11s seed 42


completed yolo11s seed 42: tail AP50=0.6461, mAP50=0.7304

[2026-11-19T07:34:04.818354+00:00] yolo11s seed 43
completed yolo11s seed 43: tail AP50=0.6622, mAP50=0.7350



[2026-11-19T07:34:05.007076+00:00] yolo11s seed 44
completed yolo11s seed 44: tail AP50=0.6432, mAP50=0.7240



Saved 12 validation records.
{
  "selection_split": "validation",
  "selection_rule": "highest mean minority macro AP50 across seeds 42, 43, 44; tie-break by mean overall mAP50",
  "minority_classes": [
    "crazing",
    "pitted_surface",
    "rolled-in_scale",
    "scratches"
  ],
  "selected_model": "yolo26n",
  "selected_pretrained_checkpoint": "yolo26n.pt",
  "selected_pretrained_sha256": "9b09cc8bf347f0fc8a5f7657480587f25db09b34bf33b0652110fb03a8ad4fef",
  "mean_minority_macro_AP50": 0.679027937344546,
  "mean_validation_mAP50": 0.748896418541454,
  "test_split_read": false
}



model_name,seeds,precision_mean,precision_sd,recall_mean,recall_sd,mAP50_mean,mAP50_sd,mAP50_95_mean,mAP50_95_sd,global_F1_mean,global_F1_sd,macro_F1_mean,macro_F1_sd,minority_macro_AP50_mean,minority_macro_AP50_sd,minority_macro_recall_mean,minority_macro_recall_sd,minority_macro_F1_mean,minority_macro_F1_sd,parameters_mean,parameters_sd,checkpoint_bytes_mean,checkpoint_bytes_sd,training_seconds_mean,training_seconds_sd,validation_inference_ms_per_image_mean,validation_inference_ms_per_image_sd
yolo26n,"42,43,44",0.7057581389329414,0.020781801044822548,0.6899206440100958,0.024609579836060477,0.748896418541454,0.007960199656907533,0.43857659475885236,0.0003900086702886653,0.6972612696703031,0.0036643217167709233,0.6898378123120688,0.008882475611912119,0.679027937344546,0.012135983973670699,0.6153782333393315,0.026317876732498042,0.6264098653328395,0.008134039747373405,2536316.0,0.0,5457989.0,0.0,284.80333333333334,0.3133565594228721,1.5642741854650208,0.01736141908243981
yolo11n,"42,43

## 3. Input-size ablation
The architecture family is selected by validation at 640 pixels. Compare the same pretrained model at 320 pixels using the same three seeds, train/validation split, batch size and epoch cap. This is a separate compute/performance ablation and does not enter the final candidate ranking.


In [3]:
resolution_command = [sys.executable, '-m', 'src.atdl.run_resolution_ablation']
subprocess.run(resolution_command, cwd=PROJECT_ROOT, check=True)
resolution_summary = PROJECT_ROOT / 'results/comprehensive/resolution_ablation_summary.csv'
print(resolution_summary.read_text())


Resolution ablation: yolo26n, imgsz=320, seed=42
tail AP50=0.6399; mAP50=0.7183; training=179.0s

Resolution ablation: yolo26n, imgsz=320, seed=43
tail AP50=0.6669; mAP50=0.7392; training=178.8s

Resolution ablation: yolo26n, imgsz=320, seed=44
tail AP50=0.6370; mAP50=0.7073; training=179.7s


{
  "model_name": "yolo26n",
  "selection_split_for_model_family": "validation",
  "comparison": "paired seeds; same COCO pretrained weights, dataset split, epochs, batch, optimizer and patience; only imgsz changes from 640 to 320",
  "imgsz_640_reference": {
    "mean_tail_macro_AP50": 0.679027937344546,
    "sd_tail_macro_AP50": 0.012135983973670699,
    "mean_mAP50": 0.748896418541454,
    "sd_mAP50": 0.007960199656907533,
    "mean_mAP50_95": 0.43857659475885236,
    "sd_mAP50_95": 0.0003900086702886653,
    "mean_inference_ms_per_image": 1.5642741854650208,
    "sd_inference_ms_per_image": 0.01736141908243981,
    "mean_training_seconds": 284.80333333333334
  },
  "imgsz_320": {
    "n_seeds": 3,
    "mean_tail_macro_AP50": 0.6479071548794405,
    "sd_tail_macro_AP50": 0.016481038258383617,
    "mean_mAP50": 0.7215876433103897,
    "sd_mAP50": 0.01619922783484536,
    "mean_mAP50_95": 0.4193814146452301,
    "sd_mAP50_95": 0.004853433453336451,
    "mean_inference_ms_per_image": 0

model_name,n_seeds,imgsz640_mean_tail_macro_AP50,imgsz640_sd_tail_macro_AP50,imgsz640_mean_mAP50,imgsz640_sd_mAP50,imgsz640_mean_mAP50_95,imgsz640_sd_mAP50_95,imgsz640_mean_inference_ms_per_image,imgsz640_sd_inference_ms_per_image,imgsz640_mean_training_seconds,imgsz320_n_seeds,imgsz320_mean_tail_macro_AP50,imgsz320_sd_tail_macro_AP50,imgsz320_mean_mAP50,imgsz320_sd_mAP50,imgsz320_mean_mAP50_95,imgsz320_sd_mAP50_95,imgsz320_mean_inference_ms_per_image,imgsz320_sd_inference_ms_per_image,imgsz320_mean_training_seconds,delta_minority_macro_ap50,delta_sd_minority_macro_ap50,delta_minority_macro_recall,delta_sd_minority_macro_recall,delta_minority_macro_f1,delta_sd_minority_macro_f1,delta_metrics/mAP50(B),delta_sd_metrics/mAP50(B),delta_metrics/mAP50-95(B),delta_sd_metrics/mAP50-95(B),delta_metrics/precision(B),delta_sd_metrics/precision(B),delta_metrics/recall(B),delta_sd_metrics/recall(B),delta_metrics/F1(B),delta_sd_metrics/F1(B),delta_validation_inference_ms_per_image,delta_sd_validatio

## 3. Full-data imbalance interventions
The selected architecture is fixed from the model-transfer validation comparison. All three treatments retain all 1,257 original training images. Repeat sampling uses square-root inverse instance frequency and deterministic image-level stochastic rounding. Targeted augmentation adds horizontal flips and 90-degree rotations with analytically transformed boxes and mild brightness shifts to tail-containing images. The combined treatment applies both. Each run targets 62,850 image presentations (1,257 × 50); epochs are adjusted for the expanded manifest and early stopping is disabled. Validation/test are unchanged.

In [4]:
intervention_command = [sys.executable, '-m', 'src.atdl.run_full_data_interventions']
subprocess.run(intervention_command, cwd=PROJECT_ROOT, check=True)
intervention_summary = PROJECT_ROOT / 'results/comprehensive/full_data_intervention_summary.csv'
print(intervention_summary.read_text())


Building full_repeat (all Phase 2 training images retained)
1543 training image presentations per epoch

[full_repeat] seed 42: yolo26n
tail AP50=0.6475; mAP50=0.7299

[full_repeat] seed 43: yolo26n
tail AP50=0.6566; mAP50=0.7362

[full_repeat] seed 44: yolo26n
tail AP50=0.6698; mAP50=0.7375

Building full_targeted_aug (all Phase 2 training images retained)
2937 training image presentations per epoch

[full_targeted_aug] seed 42: yolo26n
tail AP50=0.6797; mAP50=0.7502

[full_targeted_aug] seed 43: yolo26n
tail AP50=0.6785; mAP50=0.7442

[full_targeted_aug] seed 44: yolo26n


tail AP50=0.7014; mAP50=0.7619

Building full_combined (all Phase 2 training images retained)
3223 training image presentations per epoch

[full_combined] seed 42: yolo26n
tail AP50=0.7060; mAP50=0.7611

[full_combined] seed 43: yolo26n
tail AP50=0.6990; mAP50=0.7593

[full_combined] seed 44: yolo26n
tail AP50=0.6901; mAP50=0.7512


treatment,seeds,minority_macro_ap50_mean,minority_macro_ap50_sd,minority_macro_recall_mean,minority_macro_recall_sd,minority_macro_f1_mean,minority_macro_f1_sd,macro_f1_mean,macro_f1_sd,training_seconds_mean,training_seconds_sd,metrics/mAP50(B)_mean,metrics/mAP50(B)_sd,metrics/mAP50-95(B)_mean,metrics/mAP50-95(B)_sd,metrics/F1(B)_mean,metrics/F1(B)_sd,metrics/precision(B)_mean,metrics/precision(B)_sd,metrics/recall(B)_mean,metrics/recall(B)_sd,validation_inference_ms_mean,validation_inference_ms_sd
full_combined,"42,43,44",0.6983644177664938,0.00799500416095242,0.651429774990157,0.027480965565729997,0.6491685104705598,0.005003603130413626,0.6989742043502137,0.003286848203982006,267.46566666666666,0.812536973517731,0.7572155504049625,0.005254155399324299,0.44024466808939444,0.0037158620225807445,0.7039628481629946,0.006359578962117872,0.6970187718363179,0.03510292544546597,0.7126908608539225,0.023224847888622516,1.5647301315720938,0.01111486142242333
full_targeted_aug,"42,43,44",0.68653

## 4. Validation-only selection and figures
This compares architecture and intervention groups by mean minority macro AP50 over three seeds, with overall mAP50 as the tie-break. The test split remains unread in this cell.

In [5]:
selection_command = [sys.executable, '-m', 'src.atdl.finalize_evaluation', '--select-only']
subprocess.run(selection_command, cwd=PROJECT_ROOT, check=True)
selection_path = PROJECT_ROOT / 'results/comprehensive/final_model_selection.json'
selection = json.loads(selection_path.read_text())
print(json.dumps(selection, indent=2))

{
  "selection": {
    "selected_candidate_group": "full_combined",
    "selected_seed": 42,
    "checkpoint": "outputs/checkpoints/phase8/interventions/full_combined/seed_42/weights/best.pt",
    "checkpoint_sha256": "4a28443feef68baf1b744f6896c99c6ee1904ff220e73f3fe8b616b1f8b9f97a",
    "selection_split": "validation",
    "selection_rule": "highest mean minority macro AP50 across three seeds; tie-break by mean overall mAP50; choose the seed-specific checkpoint by validation minority AP50 then mAP50",
    "candidate_validation_summary": [
      {
        "candidate": "full_combined",
        "n_seeds": 3,
        "seeds": "42,43,44",
        "minority_macro_AP50_mean": 0.6983644177664938,
        "minority_macro_AP50_sd": 0.00799500416095242,
        "minority_macro_recall_mean": 0.651429774990157,
        "minority_macro_recall_sd": 0.027480965565729997,
        "minority_macro_F1_mean": 0.6491685104705598,
        "minority_macro_F1_sd": 0.005003603130413626,
        "precision_mea

{
  "selected_candidate_group": "full_combined",
  "selected_seed": 42,
  "checkpoint": "outputs/checkpoints/phase8/interventions/full_combined/seed_42/weights/best.pt",
  "checkpoint_sha256": "4a28443feef68baf1b744f6896c99c6ee1904ff220e73f3fe8b616b1f8b9f97a",
  "selection_split": "validation",
  "selection_rule": "highest mean minority macro AP50 across three seeds; tie-break by mean overall mAP50; choose the seed-specific checkpoint by validation minority AP50 then mAP50",
  "candidate_validation_summary": [
    {
      "candidate": "full_combined",
      "n_seeds": 3,
      "seeds": "42,43,44",
      "minority_macro_AP50_mean": 0.6983644177664938,
      "minority_macro_AP50_sd": 0.00799500416095242,
      "minority_macro_recall_mean": 0.651429774990157,
      "minority_macro_recall_sd": 0.027480965565729997,
      "minority_macro_F1_mean": 0.6491685104705598,
      "minority_macro_F1_sd": 0.005003603130413626,
      "precision_mean": 0.6970187718363179,
      "precision_sd": 0.03510

## 5. Final held-out test — run once
This cell evaluates the selected checkpoint on the test split exactly once. It also calibrates per-class confidence thresholds on validation only, then applies those fixed thresholds to test predictions for an operating-point error analysis. Do not rerun after reviewing results.

In [6]:
final_test_command = [sys.executable, '-m', 'src.atdl.finalize_evaluation']
subprocess.run(final_test_command, cwd=PROJECT_ROOT, check=True)
test_jsons = sorted((PROJECT_ROOT / 'results/comprehensive/test_evaluations').glob('*/*_test_metrics.json'))
assert test_jsons, 'Final test metrics were not written.'
test_results = json.loads(test_jsons[-1].read_text())
print(json.dumps(test_results, indent=2))

{
  "selection": {
    "selected_candidate_group": "full_combined",
    "selected_seed": 42,
    "checkpoint": "outputs/checkpoints/phase8/interventions/full_combined/seed_42/weights/best.pt",
    "checkpoint_sha256": "4a28443feef68baf1b744f6896c99c6ee1904ff220e73f3fe8b616b1f8b9f97a",
    "selection_split": "validation",
    "selection_rule": "highest mean minority macro AP50 across three seeds; tie-break by mean overall mAP50; choose the seed-specific checkpoint by validation minority AP50 then mAP50",
    "candidate_validation_summary": [
      {
        "candidate": "full_combined",
        "n_seeds": 3,
        "seeds": "42,43,44",
        "minority_macro_AP50_mean": 0.6983644177664938,
        "minority_macro_AP50_sd": 0.00799500416095242,
        "minority_macro_recall_mean": 0.651429774990157,
        "minority_macro_recall_sd": 0.027480965565729997,
        "minority_macro_F1_mean": 0.6491685104705598,
        "minority_macro_F1_sd": 0.005003603130413626,
        "precision_mea

{
  "experiment": "full_combined_seed42_test_once",
  "candidate": "full_combined",
  "seed": 42,
  "checkpoint": "outputs/checkpoints/phase8/interventions/full_combined/seed_42/weights/best.pt",
  "checkpoint_sha256": "4a28443feef68baf1b744f6896c99c6ee1904ff220e73f3fe8b616b1f8b9f97a",
  "selection_split": "validation",
  "evaluated_split": "test",
  "test_evaluation_count": 1,
  "test_split_was_read_for_selection": false,
  "global_metrics": {
    "metrics/precision(B)": 0.7290016851871034,
    "metrics/recall(B)": 0.7451771747178858,
    "metrics/mAP50(B)": 0.7842156176189748,
    "metrics/mAP50-95(B)": 0.4709893038609485,
    "fitness": 0.4709893038609485,
    "metrics/F1(B)": 0.7370006868329597
  },
  "test_speed_ms_per_image": {
    "preprocess": 1.0980632608700769,
    "inference": 1.9343096956524042,
    "loss": 0.001107456521334614,
    "postprocess": 1.363377094202096
  },
  "per_class_ap_metrics": {
    "crazing": {
      "precision": 0.4763613244126414,
      "recall": 0.354

## 6. Interpretation and deliverables
Review `reports/FINAL_EXPERIMENT_REPORT.md` and the figures in `results/comprehensive/figures/`. The report distinguishes historical one-seed Phase 5 outcomes from the repeated, equal-exposure Phase 8 results. It records gains and trade-offs as observed; a candidate is not claimed to outperform unless validation and final test evidence support that claim.

## 7. Generate the final report files
After the metrics and visualizations are saved, build the project report in Markdown and Word format. The editable presentation is generated separately from the same result records with `src/atdl/build_final_slides.mjs`.


In [7]:
report_command = [sys.executable, '-m', 'src.atdl.build_final_report']
subprocess.run(report_command, cwd=PROJECT_ROOT, check=True)
docx_command = [sys.executable, '-m', 'src.atdl.build_report_docx']
subprocess.run(docx_command, cwd=PROJECT_ROOT, check=True)
print('Markdown report:', PROJECT_ROOT / 'reports/FINAL_EXPERIMENT_REPORT.md')
print('Word report:', PROJECT_ROOT / 'reports/FINAL_EXPERIMENT_REPORT.docx')

/home/vsu/Desktop/atdl_final_project/reports/FINAL_EXPERIMENT_REPORT.md


/home/vsu/Desktop/atdl_final_project/reports/FINAL_EXPERIMENT_REPORT.docx
Markdown report: /home/vsu/Desktop/atdl_final_project/reports/FINAL_EXPERIMENT_REPORT.md
Word report: /home/vsu/Desktop/atdl_final_project/reports/FINAL_EXPERIMENT_REPORT.docx
